# MLflow 实验（Day 5 · 15-MLFlow）

> 这是 `MLFlowSetup.ipynb` 的**修正版**，适配你本机的 **MLflow 3.17.1**。
> 原始 notebook 有几处代码在 3.x 下会直接报错，每处修改都写了原因。

**运行前**：右上角选择内核 → `06_MLflow\venv`（Python 3.12.10）。

**建议按顺序逐格运行**，不要一次性 Run All —— 每格下面都有自检提示，看懂了再往下。
## 单元格依赖表（哪格能跳？）

Notebook 的单元格**共享同一个内核状态**，所以顺序很重要。

| 代码格 | 做什么 | 依赖 | 能单独跑吗 |
|---|---|---|---|
| 0 | 确认内核 | — | ✅ |
| 1 | 确认版本 | — | ✅ |
| 2 | **加载数据** | — | ✅，但后面 5 格都依赖它 |
| 3 | 创建 experiment | — | ✅ |
| 4 | 定义训练函数 | 格 2 | ✅ 定义，调用需格 2 |
| 5 | 跑 3 个配置 | 格 2 + 3 + 4 | ❌ |
| 6 | search_runs 表格 | 格 3 | ❌ |
| 7 | 训练并注册 | 格 2 + 3 | ❌ |
| 8 | 列出 version | 格 7 | ❌ |
| 9 | 从 registry 加载 | 格 8 | ❌ |
| 10 | 设 alias champion | 格 8 + 9 | ❌ |
| 11 | 打印 serve URI | 格 9 + 10 | ❌ |
| 12 | 调服务验证 | 格 10 + 11，且服务在跑 | ❌ |
| 13 | 汇总输出 | 格 6 + 8 + 9 | ❌ |

**结论**：只有格 0–3 能随便单独跑；从格 5 起必须按顺序。

> 跳过「加载数据」格直接跑训练，会得到 `NameError: name 'X_train' is not defined`。
> 不必纠结哪格能跳 —— 每格都很快，**按顺序全跑一遍最省事**。

> 💡 **tracking 存储提醒**：本 notebook 用默认 tracking URI，数据库会建在
> 「notebook 所在目录 / 当前工作目录」。所以 `mlflow ui` 前要 `cd` 到同一个目录，
> 否则会看到空的 UI。


## 0. 确认内核对不对

先跑这一格。如果打印的路径里**没有** `06_MLflow\venv`，说明内核选错了，右边换内核再跑。

In [ ]:
# 依赖：无。纯 `import sys`，检查内核路径。

import sys

print("python :", sys.version.split()[0])
print("prefix :", sys.prefix)

if "06_MLflow" not in sys.prefix:
    print("\n>>> 内核选错了！右上角切换成 06_MLflow\\venv 再重跑本格。")
else:
    print("\n>>> 内核正确。")

## 1. 装包与版本确认

原 notebook 写的是 `! pip3 install sklearn` —— **不要这么装**。`sklearn` 是个空壳包，真正的库叫 `scikit-learn`。

你环境里已经有了，所以这格只是确认，不用真的装。

In [ ]:
# 依赖：无。只 import sklearn / mlflow，打印版本。

# 需要装的时候才取消下面两行的注释
# ! pip install scikit-learn pandas
# ! pip install dagshub

import sklearn
import mlflow

print("scikit-learn :", sklearn.__version__)
print("mlflow       :", mlflow.__version__)
print("tracking uri :", mlflow.get_tracking_uri())

### 🚨 3.x 与讲义的第一个大差异

你装的是 **MLflow 3.17.1**，而讲义和原 notebook 是按 **2.x** 写的。

**好消息**：`mlflow.sklearn.log_model(...)`、`registered_model_name=`、`mlflow.pyfunc.load_model("models:/...")` 这些核心用法在 3.x 里**依然可用**，所以实验能照常做。

**要注意的**：3.x 里 `log_model` 的第二个位置参数不再是「artifact 路径」，而被当成**模型名 `name`**，并且模型名有严格限制 —— **不能含 `.`、`/`、`:`、`%`、引号**。

原 notebook 里 `train_and_log_model(100, 10, "random_forest_model_run_V1.0.2")` 含两个点，在 3.x 会直接抛：

```
MlflowException: Invalid model name ('random_forest_model_run_V1.0.2') provided.
Model name must be a non-empty string and cannot contain the following characters: ('/', ':', '.', '%', '"', "'")
```

下面我把 run 名和 artifact 名**彻底分开**，这样最清晰，也最贴近官方文档。

## 2. 加载数据

用 `load_digits()`：1797 张 8×8 的手写数字图，展平成 64 个特征，10 个类别。
`random_state=42` 是为了**每次运行切分都一样**，这样不同 run 的 accuracy 才可比 —— 这是个关键细节。

In [ ]:
# 依赖：无。**但后面 5 格都依赖它定义的 X_train / y_train / X_test / y_test。**

from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

data = load_digits()
print("样本数 :", data.data.shape)
print("类别   :", list(data.target_names))

X_train, X_test, y_train, y_test = train_test_split(
    data.data, data.target, test_size=0.2, random_state=42
)
print("训练集 :", X_train.shape, " 测试集 :", X_test.shape)

assert data.data.shape[1] == 64, "digits 应该是 64 个特征，serve 那一节的请求体依赖这个数字"

## 3. 先「创建 / 选中」experiment

**这一格就是讲义说的 "how to create a new experiment"。**

原 notebook 的注释说 `mlflow.set_experiment(name)` 会自动创建，这没错。但我这里显式写一遍 `create_experiment` + `set_experiment`，**因为 3.x 有一个坑**：`set_experiment()` 在切换 experiment 时会去找名为 `Default` 的 experiment，找不到会报错。显式创建最稳，而且你能在 UI 里看到它。

再次运行这一格不会报错（已存在就复用）。

In [ ]:
# 依赖：无（需上一格已 import mlflow）。创建并选中 experiment。

EXPERIMENT_NAME = "RandomForest_V1.0.1"

existing = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
if existing is None:
    experiment_id = mlflow.create_experiment(EXPERIMENT_NAME)
    print(f"新建 experiment '{EXPERIMENT_NAME}' -> id={experiment_id}")
else:
    experiment_id = existing.experiment_id
    print(f"复用已有 experiment '{EXPERIMENT_NAME}' -> id={experiment_id}")

mlflow.set_experiment(EXPERIMENT_NAME)
mlflow.set_tracking_uri(mlflow.get_tracking_uri())  # 确认仍是本地 sqlite

## 4. 训练 + 记录（核心格子）

这一格对应讲义的两个知识点：**log parameters** 和 **log metrics**，外加记录模型。

`with mlflow.start_run():` 里的所有 log 调用，都归属于这一次运行。

### 相比原 notebook 我改了三处

| 改动 | 原因 |
|---|---|
| `start_run(run_name=...)` 显式给 run 起名 | UI 里一眼能分辨哪个是哪个，不用靠 run_id |
| `log_model(model, name="model", ...)` | 原写法 `log_model(model, run_name)` 在 3.x 会把 run 名当**模型名**校验，含点就崩 |
| 加 `skops_trusted_types=[...]` | **3.17 的关键坑**，见下面说明 |

### 为什么需要 `skops_trusted_types`

MLflow 3.17 默认用 **skops** 序列化 sklearn 模型，skops 出于安全考虑会拒绝加载含有「不受信任类型」的文件。而 `RandomForestClassifier` 内部必然含 `sklearn.tree._tree.Tree`，于是**原 notebook 的代码会直接报错**：

```
UntrustedTypesFoundException: Untrusted types found in the file: ['sklearn.tree._tree.Tree']
MlflowException: The saved sklearn model references untrusted types...
```

因为这个模型是**你自己刚训练出来的**，信任它是合理的，所以显式声明 `skops_trusted_types=["sklearn.tree._tree.Tree"]`。

> 换别的模型（如 `LogisticRegression`）可能不需要这一项；报错时按提示把它列出的类型加进去即可。

In [ ]:
# 依赖：需要「加载数据」格已跑（函数体里用到 X_train 等）。本格只是定义函数，不训练。

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score


def train_and_log_model(n_estimators, max_depth, run_name):
    """训练一个模型，把参数/指标/模型都记录到 MLflow，返回 (run_id, accuracy)。"""
    with mlflow.start_run(run_name=run_name) as run:
        model = RandomForestClassifier(
            n_estimators=n_estimators, max_depth=max_depth, random_state=42
        )
        model.fit(X_train, y_train)

        accuracy = accuracy_score(y_test, model.predict(X_test))
        train_accuracy = model.score(X_train, y_train)

        # --- 记录参数 ---
        mlflow.log_param("n_estimators", n_estimators)
        mlflow.log_param("max_depth", max_depth)
        mlflow.log_param("test_size", 0.2)

        # --- 记录指标 ---
        mlflow.log_metric("accuracy", accuracy)
        mlflow.log_metric("train_accuracy", train_accuracy)
        # 训练集与测试集准确率的差距 = 过拟合的粗略信号
        mlflow.log_metric("overfit_gap", train_accuracy - accuracy)

        # --- 记录标签（方便日后检索/筛选）---
        mlflow.set_tag("model_type", "RandomForestClassifier")
        mlflow.set_tag("dataset", "sklearn.load_digits")

        # --- 记录模型本体 ---
        model_info = mlflow.sklearn.log_model(
            model,
            name="model",                                    # 3.x 用 name（原 notebook 的 artifact_path 已弃用）
            skops_trusted_types=["sklearn.tree._tree.Tree"],  # 3.17 必须，否则报 untrusted types
        )

        print(
            f"run={run_name:34s} run_id={run.info.run_id}  "
            f"acc={accuracy:.4f}  train={train_accuracy:.4f}  gap={train_accuracy - accuracy:+.4f}"
        )
        print(f"    模型已记录: {model_info.model_uri}")
        return run.info.run_id, accuracy, model


## 5. 正式跑三个配置

原 notebook 只跑两个。我多加了第三个**故意很差的配置**，因为有了对比才能说明 MLflow 的价值 —— 尤其是「哪个配置过拟合最严重」。

In [ ]:
# 依赖：需要 2（数据）+ 3（experiment 已选中）+ 4（train_and_log_model 已定义）。**不能跳。**

print("Training and Logging Model 1...")
run_id_1, acc_1, model_1 = train_and_log_model(20, 5, "rf_run_1_n20_d5")

print("\nTraining and Logging Model 2...")
run_id_2, acc_2, model_2 = train_and_log_model(100, 10, "rf_run_2_n100_d10")

print("\nTraining and Logging Model 3 (故意很差)...")
run_id_3, acc_3, model_3 = train_and_log_model(5, 2, "rf_run_3_n5_d2")

## 6. 用代码确认记录成功

不用急着开 UI，先用 `search_runs` 把结果拉出来看一眼。

**自检点**：应该看到 3 行，accuracy 从高到低是 model 2 > model 1 > model 3。最差那个（n5/d2）的 `overfit_gap` 应该最小。

In [ ]:
# 依赖：需要 3 定义的 experiment_id。**不能跳。**

import pandas as pd

pd.set_option("display.width", 200)

runs_df = mlflow.search_runs(
    experiment_ids=[experiment_id],
    order_by=["metrics.accuracy DESC"],
)

print(
    runs_df[
        [
            "tags.mlflow.runName",
            "params.n_estimators",
            "params.max_depth",
            "metrics.accuracy",
            "metrics.train_accuracy",
            "metrics.overfit_gap",
            "run_id",
        ]
    ].to_string(index=False)
)

assert len(runs_df) >= 3, "应该至少有 3 个 run"
print(f"\n共 {len(runs_df)} 个 run，experiment_id={experiment_id}")

## 7. 打开 MLflow UI

**在 VS Code 的终端里**（不是在 notebook 单元格里）执行：

```powershell
cd E:\dsh\academic\software\06_MLflow
E:\dsh\academic\software\06_MLflow\venv\Scripts\mlflow.exe ui
```

`mlflow ui` 默认读**当前目录**，所以要 `cd` 到存放 `mlruns\` 和 `mlflow.db` 的那个目录 —— 也就是 `06_MLflow`。

然后浏览器打开 **http://127.0.0.1:5000**。

**自检点**：
- Experiments 里能看到 `RandomForest_V1.0.1`，里面有 3 个 run
- 点开任一 run，Parameters / Metrics / Artifacts 三部分都有内容
- 点 Artifacts 能看到 `model` 目录（里面有 `MLmodel`、`model.pkl` 等文件）
- 上方出现 **Models** 页签（等第 8 节注册之后才会有东西）

> 如果 5000 端口被占用：`mlflow ui --port 5001`
> 停服务：终端里 Ctrl+C

## 8. Model Registry：把模型注册进注册表

**这是讲义明确要求、但原 notebook 完全没写的部分。**

Tracking 只是「记录」，Registry 才是「**版本化管理 + 标记哪个版本是生产版**」。

```
Registered Model: RandomForest_Digits
  ├── Version 1   accuracy=0.9389
  ├── Version 2   accuracy=0.9722   ← alias: champion（线上用这个）
  └── Version 3   accuracy=0.5917
```

做法很简单：`log_model` 时多传一个 `registered_model_name`，每次同名就会自动产生一个新 Version。

> **3.x 的差异**：3.x 取消了 2.x 的 stage（Staging/Production/Archived），改用 **alias**（别名）。所以标记「生产版」要用 `MlflowClient.set_registered_model_alias("名字", "champion", version)`。

In [ ]:
# 依赖：需要 2（数据）+ 3（experiment 已选中）。**不能跳。**

from mlflow import MlflowClient

client = MlflowClient()
REGISTERED_NAME = "RandomForest_Digits"


def train_register(n_estimators, max_depth, run_name):
    """训练 + 记录 + 注册到 Model Registry。"""
    with mlflow.start_run(run_name=run_name) as run:
        model = RandomForestClassifier(
            n_estimators=n_estimators, max_depth=max_depth, random_state=42
        )
        model.fit(X_train, y_train)
        accuracy = accuracy_score(y_test, model.predict(X_test))

        mlflow.log_param("n_estimators", n_estimators)
        mlflow.log_param("max_depth", max_depth)
        mlflow.log_metric("accuracy", accuracy)
        mlflow.log_metric("train_accuracy", model.score(X_train, y_train))

        mlflow.sklearn.log_model(
            model,
            name="model",
            registered_model_name=REGISTERED_NAME,          # ← 注册进 Registry 的关键
            skops_trusted_types=["sklearn.tree._tree.Tree"],
        )
        print(f"run={run_name:26s} acc={accuracy:.4f} 已注册，run_id={run.info.run_id}")
        return run.info.run_id, accuracy


reg_run_id_1, reg_acc_1 = train_register(20, 5, "registry_v1_n20_d5")
reg_run_id_2, reg_acc_2 = train_register(100, 10, "registry_v2_n100_d10")

### 看看 Registry 里现在有什么

**自检点**：`RandomForest_Digits` 应该有 2 个 version。

In [ ]:
# 依赖：需要 7 已经注册过模型。**不能跳。**

print(f"'{REGISTERED_NAME}' 的所有 version：\n")
for mv in client.search_model_versions(f"name='{REGISTERED_NAME}'"):
    print(f"  version {mv.version:>2}  | run_id={mv.run_id}  | status={mv.status}")

## 9. 从 Registry 加载模型回来

**这也是讲义明确要求、原 notebook 没写的部分。**

能加载回来，才证明「注册」是真的有用的 —— 别人（或你明天）不需要你的 notebook，只要有 registry 名称就能复现预测。

**自检点**：`预测一致: True`。这意味着从 registry 加载的模型，预测结果和内存里那个模型**完全一样**。

In [ ]:
# 依赖：需要 8 得到的 latest_version 与 REGISTERED_NAME。**不能跳。**

# 先取最新 version 号
versions = client.search_model_versions(f"name='{REGISTERED_NAME}'")
latest_version = max(int(v.version) for v in versions)
print("最新 version :", latest_version)

# 方式一：用版本号
loaded_v = mlflow.pyfunc.load_model(f"models:/{REGISTERED_NAME}/{latest_version}")
pred_v = loaded_v.predict(X_test)

# 方式二：用 latest
loaded_latest = mlflow.pyfunc.load_model(f"models:/{REGISTERED_NAME}/latest")
pred_latest = loaded_latest.predict(X_test)

# 内存里重新训一个一样的模型做对照
ref = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
ref.fit(X_train, y_train)
pred_ref = ref.predict(X_test)

print("registry(版本号) 预测一致:", bool((pred_v == pred_ref).all()))
print("registry(latest) 预测一致:", bool((pred_latest == pred_ref).all()))
print("前 10 个预测:", pred_v[:10].tolist())
print("真实标签   :", y_test[:10].tolist())

assert (pred_v == pred_ref).all(), "从 registry 加载的模型预测结果不一致，检查一下"

### 9b. 用 alias 标记「生产版」（3.x 的 stage 替代品）

把表现最好的那个版本打上 `champion` 别名，之后别人就可以用 `models:/RandomForest_Digits@champion` 拿到它，**不用关心版本号变了没有**。

In [ ]:
# 依赖：需要 8（best_version）+ 9 的加载方式。**不能跳。**

# 找出 accuracy 最高的那个 version
best_version, best_acc = None, -1.0
for mv in client.search_model_versions(f"name='{REGISTERED_NAME}'"):
    run = client.get_run(mv.run_id)
    acc = run.data.metrics.get("accuracy", -1.0)
    if acc > best_acc:
        best_version, best_acc = mv.version, acc

client.set_registered_model_alias(REGISTERED_NAME, "champion", best_version)
print(f"alias 'champion' -> version {best_version} (accuracy={best_acc:.4f})")

champion = mlflow.pyfunc.load_model(f"models:/{REGISTERED_NAME}@champion")
print("通过 @champion 加载并预测:", champion.predict(X_test[:5]).tolist())
print("对应真实标签            :", y_test[:5].tolist())

## 10. 本地部署（Serve）

**这一步在终端里做，不是在 notebook 里。**

### 10.1 拿到 model URI

下面这格会打印出可以直接给 `mlflow models serve` 用的 URI，把它复制下来。

In [ ]:
# 依赖：需要 9（best_version、versions）+ 10 定义的 ref 与 latest_version。**不能跳。**

# 用 registry 里的 champion（最推荐：稳定、不用记 run_id）
print("给 mlflow models serve 用的 URI（三选一）：\n")
print(f"  1) models:/{REGISTERED_NAME}@champion")
print(f"  2) models:/{REGISTERED_NAME}/{latest_version}")
print(f"  3) runs:/{reg_run_id_2}/model")

### 10.2 启动服务（在 VS Code 终端里执行）

```powershell
cd E:\dsh\academic\software\06_MLflow
E:\dsh\academic\software\06_MLflow\venv\Scripts\mlflow.exe models serve `
  -m "models:/RandomForest_Digits@champion" -p 5003 --env-manager local
```

**关于 `--env-manager local`**：
- 不加这个参数，MLflow 会尝试用 virtualenv/conda **重建一个环境**来跑模型，你环境里没有 virtualenv，会报 `ModuleNotFoundError: No module named 'virtualenv'`（讲义里提到的就是这个坑）。
- 这个实验里模型就跑在本地，用 `local` 直接复用当前环境，最省事。
- 如果非要走标准流程，就 `pip install virtualenv` 然后把 `--env-manager local` 去掉。

**自检点**：终端出现 `Listening at: http://127.0.0.1:5003`，并且**这个终端不要关**。

### 10.3 调用它验证

**另开一个终端**，或者直接在 notebook 里跑下一格（服务必须还在运行）。

MLflow 的 pyfunc 服务接收 `dataframe_split` 格式：64 个特征对应 8×8 像素。

In [ ]:
import requests

SERVE_URL = "http://127.0.0.1:5003/invocations"

# 拿一行真实数据（第 0 个测试样本）
row = X_test[0].tolist()
body = {
    "dataframe_split": {
        "columns": [f"f{i}" for i in range(64)],
        "data": [row],
    }
}

try:
    resp = requests.post(SERVE_URL, json=body, timeout=30)
    resp.raise_for_status()
    served_pred = resp.json()["predictions"][0]
    local_pred = int(ref.predict(X_test[:1])[0])
    print("服务返回   :", served_pred)
    print("本地模型   :", local_pred)
    print("真实标签   :", int(y_test[0]))
    print("\n自检:", "一致 ✅" if int(served_pred) == local_pred else "不一致 ❌")
except requests.exceptions.ConnectionError:
    print("连不上服务。检查：\n"
          "  1) 10.2 的 mlflow models serve 那个终端还在跑吗？\n"
          "  2) 端口是 5003 吗？\n"
          "  3) 是否被防火墙拦了？")

## 11. 收尾：把这次实验的信息记下来

跑完这一格，把输出粘到 `mlflow-lab\lab-notes.md` 里，就完成了讲义要求的第 1–3 项（安装、实验、看结果）。

In [ ]:
print("=" * 60)
print("实验汇总")
print("=" * 60)
print(f"tracking uri   : {mlflow.get_tracking_uri()}")
print(f"experiment     : {EXPERIMENT_NAME} (id={experiment_id})")
print(f"run 总数        : {len(runs_df)}")
print(f"registered name : {REGISTERED_NAME}")
print(f"version 数      : {len(versions)}   champion -> {best_version} (acc={best_acc:.4f})")
print()
print("accuracy 排名：")
for _, r in runs_df.iterrows():
    print(f"  {r['tags.mlflow.runName']:24s} acc={r['metrics.accuracy']:.4f}")
print("=" * 60)

## 12. 接下来（讲义第 4、5 项）

到这里你已经完成了讲义第 1、2、3 项。剩下两项需要你自己操作：

### 第 4 项：远程 tracking（Dagshub）

需要你自己的 Dagshub 账号，步骤在 `GUIDE.md` §5。核心就是把 tracking URI 从本地换成远程：

```python
import dagshub
dagshub.init(repo_owner='<你的用户名>', repo_name='<你的repo>', mlflow=True)
mlflow.set_experiment("RandomForest_day5_remote")   # 换个新名字，别和本地混
```

> ⚠️ token 等同密码，**不要**硬编码进 notebook 再 commit 到 GitHub。

### 第 5 项：Serve

上面 §10 已经做了。想再确认一次，就用 `models:/RandomForest_Digits/{latest_version}` 代替 `@champion` 重启服务，看输出是否一样。

### 最后交给我核对

把 `lab-notes.md` 填完，连同下面的输出发我，我帮你检查：
1. §6 的 `search_runs` 表格
2. §9 的「预测一致: True」
3. §10.3 的服务返回结果